# Principal components: a representation, not a proof

**Question.** Can a small number of orthogonal directions summarize important variation in standardized wine chemistry?

We will fit PCA through the package, inspect retained variance and loadings, visualize samples, and separate exploratory geometry from scientific interpretation. The cultivar label is used only to annotate the projection after fitting.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dsm_project.config import load_config
from dsm_project.data import load_project_dataset
from dsm_project.experiments import run_pca_example

config = load_config("configs/example.toml")
dataset = load_project_dataset(config)
frame = dataset.observations
result = run_pca_example(dataset, components=2)

print("Explained-variance ratios:", result.explained_variance_ratio)
print("Two-component retained variance:", result.explained_variance_ratio.sum())

## Variance-maximizing directions

After centering and standardizing the feature matrix, the first principal direction $v_1$ solves

$$
\max_{\lVert v\rVert_2=1}\operatorname{Var}(Xv).
$$

Later directions maximize remaining variance subject to orthogonality. Scores $Xv_j$ locate samples in the new representation; loadings describe the signed contribution of original coordinates.

Variance is not the same as predictive value, causality or scientific importance. Standardization gives each recorded feature equal variance before fitting, a choice that must be justified from units and measurement precision.

In [ ]:
loadings = pd.DataFrame(
    result.components.T,
    index=result.feature_names,
    columns=["PC1", "PC2"],
)
display(loadings.reindex(loadings["PC1"].abs().sort_values(ascending=False).index))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
loadings["PC1"].sort_values().plot.barh(ax=axes[0], color="#7570b3")
axes[0].set(title="First-component loadings", xlabel="signed loading", ylabel="")

axes[1].bar(["PC1", "PC2"], result.explained_variance_ratio, color=["#1b9e77", "#d95f02"])
axes[1].set(title="Variance retained by each component", ylabel="fraction of standardized variance")
plt.tight_layout();

In [ ]:
plt.figure(figsize=(8, 5))
for cultivar in sorted(frame["cultivar"].unique()):
    mask = frame["cultivar"].to_numpy() == cultivar
    plt.scatter(
        result.coordinates[mask, 0],
        result.coordinates[mask, 1],
        label=f"cultivar {cultivar}",
        alpha=0.75,
    )
plt.xlabel("principal component 1")
plt.ylabel("principal component 2")
plt.title("PCA fitted without cultivar labels; labels shown only for audit")
plt.legend()
plt.tight_layout();

## Reading the projection carefully

Visible separation suggests that major chemistry variation aligns partly with cultivar, but the figure is a two-dimensional shadow of a higher-dimensional sample. Apparent groups depend on scaling, sampling, outliers and the chosen components. Loading signs can flip without changing the representation, so direction signs are not inherently meaningful.

If PCA feeds a supervised model, it must be fitted inside each training fold. Fitting it on the full dataset before validation leaks holdout distribution information.

## Conclusion, limits, and student implementation

A project analysis should examine a fuller scree curve, reconstruction error, loading and subspace stability, sensitivity to scaling and outliers, and downstream value. State whether PCA is being used for visualization, compression, denoising or a fitted pipeline; those purposes require different evidence.

A student implementation should verify centering, covariance or singular-value calculations, orthonormal components, ordered explained variance, transform shape and reconstruction on a hand-computable matrix. Compare subspaces rather than raw signs when checking a trusted implementation.